# KYC SUPERHERO WORKSHOP - STEP 5: MAKER/CHECKER DASHBOARD

Final dashboard combining all maker and checker results:
- End-to-end pipeline status for each applicant
- Maker/checker agreement rates
- Discrepancy analysis
- Items requiring human escalation
- Regulatory readiness summary

In [ ]:
%%sql -r context_setup
USE ROLE KYC_WORKSHOP_ROLE;
USE DATABASE KYC_SUPERHERO_DB;
USE SCHEMA ANALYTICS;
USE WAREHOUSE KYC_WORKSHOP_WH;

In [ ]:
%%sql -r create_pipeline_status
CREATE OR REPLACE VIEW PIPELINE_STATUS AS
SELECT
  k.file_name,
  k.extracted_data:response:applicant_name::VARCHAR AS applicant_name,
  
  id_check.checker_output:check_status::VARCHAR AS id_extraction_status,
  id_check.checker_output:confidence_score::INT AS id_confidence,
  id_check.checker_output:document_authenticity::VARCHAR AS id_authenticity,
  
  kyc_check.checker_output:check_status::VARCHAR AS kyc_extraction_status,
  kyc_check.checker_output:confidence_score::INT AS kyc_confidence,
  kyc_check.checker_output:completeness_pct::INT AS kyc_completeness,
  kyc_check.checker_output:regulatory_ready::BOOLEAN AS kyc_reg_ready,
  
  xref_check.checker_output:check_status::VARCHAR AS crossref_status,
  xref_check.checker_output:confidence_score::INT AS crossref_confidence,
  xref_check.checker_output:recommended_action::VARCHAR AS crossref_recommendation,
  
  edd.edd_trigger_reason,
  edd_review.senior_review:check_status::VARCHAR AS edd_senior_decision,
  edd_review.senior_review:senior_risk_score::INT AS edd_risk_score,
  edd_review.senior_review:regulatory_compliance::VARCHAR AS edd_reg_compliance,
  
  CASE
    WHEN id_check.checker_output:check_status::VARCHAR = 'FAIL' THEN 'BLOCKED'
    WHEN kyc_check.checker_output:check_status::VARCHAR = 'FAIL' THEN 'BLOCKED'
    WHEN xref_check.checker_output:check_status::VARCHAR = 'FAIL' THEN 'BLOCKED'
    WHEN edd_review.senior_review:check_status::VARCHAR = 'REJECTED' THEN 'BLOCKED'
    WHEN edd_review.senior_review:check_status::VARCHAR = 'ESCALATE_TO_MLRO' THEN 'ESCALATED'
    WHEN id_check.checker_output:check_status::VARCHAR = 'NEEDS_REVIEW' THEN 'PENDING_REVIEW'
    WHEN kyc_check.checker_output:check_status::VARCHAR = 'NEEDS_REVIEW' THEN 'PENDING_REVIEW'
    WHEN xref_check.checker_output:check_status::VARCHAR = 'NEEDS_REVIEW' THEN 'PENDING_REVIEW'
    ELSE 'CLEARED'
  END AS overall_status

FROM KYC_SUPERHERO_DB.CURATED.EXTRACTED_KYC k

LEFT JOIN KYC_SUPERHERO_DB.CURATED.CHECKED_IDS id_check
  ON REPLACE(k.file_name, 'KYC_Form', 'ID') = REPLACE(id_check.file_name, '.png', '.pdf')

LEFT JOIN KYC_SUPERHERO_DB.CURATED.CHECKED_KYC kyc_check
  ON k.file_name = kyc_check.file_name

LEFT JOIN KYC_SUPERHERO_DB.CURATED.CHECKED_CROSSREF xref_check
  ON k.file_name = xref_check.file_name

LEFT JOIN KYC_SUPERHERO_DB.ANALYTICS.EDD_ASSESSMENT edd
  ON k.file_name = edd.file_name

LEFT JOIN KYC_SUPERHERO_DB.ANALYTICS.CHECKED_EDD edd_review
  ON k.file_name = edd_review.file_name;

In [ ]:
%%sql -r create_agreement_view
CREATE OR REPLACE VIEW MAKER_CHECKER_AGREEMENT AS

SELECT 
  'ID Card Extraction' AS pipeline_stage,
  file_name,
  maker_output:response:full_name::VARCHAR AS entity_name,
  checker_output:check_status::VARCHAR AS checker_verdict,
  checker_output:confidence_score::INT AS confidence,
  checker_output:fields_with_discrepancies::INT AS discrepancy_count,
  checker_output:document_authenticity::VARCHAR AS quality_indicator,
  checker_output:checker_notes::VARCHAR AS checker_reasoning
FROM KYC_SUPERHERO_DB.CURATED.CHECKED_IDS

UNION ALL

SELECT 
  'KYC Form Extraction' AS pipeline_stage,
  file_name,
  maker_output:response:applicant_name::VARCHAR AS entity_name,
  checker_output:check_status::VARCHAR AS checker_verdict,
  checker_output:confidence_score::INT AS confidence,
  ARRAY_SIZE(checker_output:discrepancies) AS discrepancy_count,
  CASE WHEN checker_output:regulatory_ready::BOOLEAN THEN 'REG_READY' ELSE 'NOT_READY' END AS quality_indicator,
  checker_output:checker_notes::VARCHAR AS checker_reasoning
FROM KYC_SUPERHERO_DB.CURATED.CHECKED_KYC

UNION ALL

SELECT 
  'Entity Screening' AS pipeline_stage,
  file_name,
  applicant_name AS entity_name,
  checker_output:check_status::VARCHAR AS checker_verdict,
  checker_output:confidence_score::INT AS confidence,
  ARRAY_SIZE(checker_output:potential_missed_matches) AS discrepancy_count,
  checker_output:recommended_action::VARCHAR AS quality_indicator,
  checker_output:checker_notes::VARCHAR AS checker_reasoning
FROM KYC_SUPERHERO_DB.CURATED.CHECKED_CROSSREF

UNION ALL

SELECT 
  'EDD Assessment' AS pipeline_stage,
  file_name,
  applicant_name AS entity_name,
  senior_review:check_status::VARCHAR AS checker_verdict,
  (100 - ABS(senior_review:score_variance::INT) * 10) AS confidence,
  ARRAY_SIZE(senior_review:missing_risk_factors) + ARRAY_SIZE(senior_review:additional_actions_required) AS discrepancy_count,
  senior_review:regulatory_compliance::VARCHAR AS quality_indicator,
  senior_review:final_recommendation::VARCHAR AS checker_reasoning
FROM KYC_SUPERHERO_DB.ANALYTICS.CHECKED_EDD;

In [ ]:
%%sql -r dashboard_pipeline
SELECT
  applicant_name,
  overall_status,
  id_extraction_status,
  id_confidence,
  kyc_extraction_status,
  kyc_confidence,
  crossref_status,
  crossref_recommendation,
  edd_senior_decision,
  edd_risk_score
FROM PIPELINE_STATUS
ORDER BY 
  CASE overall_status
    WHEN 'BLOCKED' THEN 1
    WHEN 'ESCALATED' THEN 2
    WHEN 'PENDING_REVIEW' THEN 3
    WHEN 'CLEARED' THEN 4
  END;

In [ ]:
%%sql -r dashboard_agreement
CREATE OR REPLACE VIEW ESCALATION_QUEUE AS
SELECT
  pipeline_stage,
  COUNT(*) AS total_checks,
  SUM(CASE WHEN checker_verdict = 'PASS' THEN 1 ELSE 0 END) AS passed,
  SUM(CASE WHEN checker_verdict = 'FAIL' THEN 1 ELSE 0 END) AS failed,
  SUM(CASE WHEN checker_verdict NOT IN ('PASS', 'FAIL') THEN 1 ELSE 0 END) AS needs_review,
  ROUND(AVG(confidence), 1) AS avg_confidence,
  SUM(discrepancy_count) AS total_discrepancies
FROM MAKER_CHECKER_AGREEMENT
GROUP BY pipeline_stage
ORDER BY pipeline_stage;

In [ ]:
%%sql -r dashboard_escalations
SELECT
  pipeline_stage,
  entity_name,
  checker_verdict,
  confidence,
  discrepancy_count,
  quality_indicator,
  checker_reasoning
FROM MAKER_CHECKER_AGREEMENT
WHERE checker_verdict IN ('FAIL', 'NEEDS_REVIEW', 'ESCALATE_TO_MLRO', 'REJECTED')
   OR confidence < 70
ORDER BY confidence ASC;

In [ ]:
%%sql -r dashboard_summary
SELECT
  overall_status,
  COUNT(*) AS applicant_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM PIPELINE_STATUS
GROUP BY overall_status
ORDER BY 
  CASE overall_status
    WHEN 'BLOCKED' THEN 1
    WHEN 'ESCALATED' THEN 2
    WHEN 'PENDING_REVIEW' THEN 3
    WHEN 'CLEARED' THEN 4
  END;